# NeuralManifoldDynamics — Quickstart

**Raw Data → NDT Feature Pipeline → coords_9d → 3D MNPS Trajectory**

This notebook runs entirely on **synthetic EEG** — no dataset download needed.  
Clone the repo, activate the venv, open the notebook, hit *Run All* — done in ≤ 5 min.

---

### The canonical MNPS pipeline (from the article)

```
features z_t  ──►  coords_9d  x_t^(9)  ──►  mnps_3d  x_t
              W_9D                     P
```

The 9D-to-3D projection is **not** a trivial equal-weight mean.  
It uses a fixed weighted projection `P` (the `v1_mapping` in config) with L2-normalised columns.

### Stratified 9D subcoordinates (contract v2.0)

| Sub-coord | Family | EEG feature |
|-----------|--------|-------------|
| `m_a` | M: macrostate | −0.5·δ − 0.5·θ |
| `m_e` | M: engagement | −1.0·α |
| `m_o` | M: organisation | +1.0·β/α |
| `d_n` | D: network binding | +1.0·γ |
| `d_l` | D: local dispersion | +1.0·Hjorth mobility |
| `d_s` | D: spectral shift | +1.0·α/θ |
| `e_e` | E: entropy | +1.0·permutation entropy |
| `e_s` | E: spectral flatness | +1.0·Hjorth complexity |
| `e_m` | E: embodied proxy | +1.0·γ (fallback) |

### Fixed 9D → 3D projection weights (`v1_mapping`)

```
m = 0.62·m_a + 0.55·m_e + 0.45·m_o      (L2-normalised at runtime)
d = 0.50·d_n + 0.82·d_l + 0.28·d_s
e = 0.85·e_e + 0.62·e_s + 0.03·e_m
```

## 0 · Environment setup

In [ ]:
import sys
from pathlib import Path

_nb_dir = Path().resolve()
REPO = _nb_dir
for _candidate in [_nb_dir, _nb_dir.parent, _nb_dir.parent.parent]:
    if (_candidate / "mndm" / "src" / "mndm").exists():
        REPO = _candidate
        break

for _src in [REPO / "mndm" / "src", REPO / "core" / "src"]:
    p = str(_src)
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Repo root : {REPO}")

## 1 · Imports

In [ ]:
import math
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

import numpy as np
import pandas as pd
from scipy.signal import welch

# MNPS pipeline modules
from mndm.projection import estimate_derivatives, build_knn_indices
from mndm.jacobian import estimate_local_jacobians
from mndm.schema import MNPSPayload, compute_meta_indices

# Plotting
try:
    import plotly.graph_objects as go
    from plotly.subplots import make_subplots
    _PLOTLY = True
except ImportError:
    import matplotlib.pyplot as plt
    from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
    _PLOTLY = False
    print("plotly not installed — falling back to matplotlib")

print(f"Ready  (backend: {'plotly' if _PLOTLY else 'matplotlib'})")

## 2 · Synthetic EEG

Five distinct brain-state regimes, 60 s each, 4 channels, 256 Hz.

In [ ]:
RNG = np.random.default_rng(42)
FS = 256; DURATION = 300; N_CH = 4
t_raw = np.linspace(0, DURATION, int(FS * DURATION), endpoint=False)

def _make_regime(n, delta=0.0, theta=0.0, alpha=0.0, beta=0.0, gamma=0.0, noise=0.3):
    t = np.arange(n) / FS
    s  = delta * np.sin(2*np.pi* 2*t + RNG.uniform(0, 6.28))
    s += theta * np.sin(2*np.pi* 6*t + RNG.uniform(0, 6.28))
    s += alpha * np.sin(2*np.pi*10*t + RNG.uniform(0, 6.28))
    s += beta  * np.sin(2*np.pi*20*t + RNG.uniform(0, 6.28))
    s += gamma * np.sin(2*np.pi*40*t + RNG.uniform(0, 6.28))
    s += noise * RNG.standard_normal(n)
    return s.astype(np.float32)

REGIMES = {
    "rest":       dict(alpha=2.0, theta=0.5, delta=0.3, beta=0.4, gamma=0.1, noise=0.2),
    "drowsy":     dict(alpha=0.8, theta=2.0, delta=2.5, beta=0.2, gamma=0.05, noise=0.15),
    "focused":    dict(alpha=0.5, theta=0.4, delta=0.2, beta=2.5, gamma=0.8, noise=0.25),
    "alert":      dict(alpha=0.3, theta=0.2, delta=0.1, beta=1.5, gamma=2.5, noise=0.3),
    "transition": dict(alpha=1.0, theta=1.0, delta=1.0, beta=1.0, gamma=0.5, noise=0.4),
}
regime_labels = list(REGIMES.keys())
block_samples = int(FS * 60)

eeg = np.zeros((N_CH, len(t_raw)), dtype=np.float32)
regime_per_sample = np.empty(len(t_raw), dtype=object)
for ch in range(N_CH):
    eeg[ch] = np.concatenate([
        (_make_regime(block_samples, **REGIMES[name]) * (1.0 + 0.1 * RNG.standard_normal()))
        for name in regime_labels
    ])
for i, name in enumerate(regime_labels):
    regime_per_sample[i * block_samples:(i + 1) * block_samples] = name

print(f"EEG shape : {eeg.shape}  ({DURATION} s, {N_CH} ch, {FS} Hz)")
print(f"Regimes   : {regime_labels}")

## 3 · Feature extraction

Bands follow `config_ingest_common_eeg.yaml` exactly:
- δ: 1–4 Hz, θ: 4–8 Hz, **α: 8–12 Hz**, β: 13–30 Hz, γ: 30–45 Hz  

Extra features required by the 9D contract v2.0:
- **Hjorth mobility and complexity** (needed for `d_l`, `e_s`)
- **Permutation entropy** order=5, delay=1 (needed for `e_e`)

In [ ]:
# ── Band definitions (exact from config_ingest_common_eeg.yaml) ──────────────
BANDS = {
    "eeg_delta": (1.0,  4.0),
    "eeg_theta": (4.0,  8.0),
    "eeg_alpha": (8.0, 12.0),   # ← 8–12, not 8–13
    "eeg_beta":  (13.0, 30.0),
    "eeg_gamma": (30.0, 45.0),
}

def _band_power(psd, freqs, lo, hi):
    mask = (freqs >= lo) & (freqs < hi)
    return float(np.mean(psd[mask])) if mask.any() else float("nan")

def _hjorth(seg):
    """Return (mobility, complexity) for a 1-D EEG segment."""
    dx  = np.diff(seg, n=1)
    ddx = np.diff(seg, n=2)
    var_x   = float(np.var(seg))
    var_dx  = float(np.var(dx))
    var_ddx = float(np.var(ddx))
    mob   = np.sqrt(var_dx  / (var_x  + 1e-12))
    comp  = np.sqrt(var_ddx / (var_dx + 1e-12)) / (mob + 1e-12)
    return float(mob), float(comp)

# Permutation entropy (order=5, delay=1, normalised) ─ vectorised per window
_ORDER = 5
_PERM_LOOKUP = {p: i for i, p in enumerate(
    __import__("itertools").permutations(range(_ORDER))
)}

def _permutation_entropy(seg, order=5, delay=1, normalize=True):
    n = len(seg) - (order - 1) * delay
    if n <= 0:
        return float("nan")
    # Build embedding matrix [n, order]
    idx   = np.arange(order) * delay
    emb   = np.stack([seg[i:i + order * delay:delay] for i in range(n)])
    ranks = np.argsort(emb, axis=1)
    counts: dict = {}
    for r in ranks:
        key = tuple(r)
        counts[key] = counts.get(key, 0) + 1
    probs = np.array(list(counts.values()), dtype=float) / n
    pe = float(-np.sum(probs * np.log2(probs + 1e-20)))
    if normalize:
        pe /= np.log2(math.factorial(order))
    return pe

# ── Sliding-window extraction ─────────────────────────────────────────────────
WIN_SEC = 8.0;  STEP_SEC = 4.0
WIN_SAMP  = int(WIN_SEC  * FS)
STEP_SAMP = int(STEP_SEC * FS)
N_WIN = (eeg.shape[1] - WIN_SAMP) // STEP_SAMP + 1

rows = []
w_start = np.zeros(N_WIN)
w_end   = np.zeros(N_WIN)
regime_per_win = []

for wi in range(N_WIN):
    s = wi * STEP_SAMP;  e = s + WIN_SAMP
    w_start[wi] = s / FS;  w_end[wi] = e / FS
    regime_per_win.append(regime_per_sample[(s + e) // 2])

    bp   = {k: 0.0 for k in BANDS}
    mob_vals, comp_vals, pe_vals = [], [], []
    total_power = 0.0

    for ch in range(N_CH):
        seg = eeg[ch, s:e].astype(np.float64)
        freqs, psd = welch(seg, fs=FS, nperseg=min(WIN_SAMP, 256))
        for band_name, (lo, hi) in BANDS.items():
            bp[band_name] += _band_power(psd, freqs, lo, hi)
        total_power += float(np.mean(psd))
        mob, comp = _hjorth(seg)
        mob_vals.append(mob);  comp_vals.append(comp)
        pe_vals.append(_permutation_entropy(seg))

    total_power = max(total_power / N_CH, 1e-20)
    # Relative band powers
    rel = {k: (v / N_CH) / total_power for k, v in bp.items()}

    a  = rel["eeg_alpha"];  b = rel["eeg_beta"];  th = rel["eeg_theta"]
    row = {**rel}
    row["eeg_beta_alpha"]        = b  / (a  + 1e-9)
    row["eeg_alpha_theta"]       = a  / (th + 1e-9)
    row["eeg_hjorth_mobility"]   = float(np.mean(mob_vals))
    row["eeg_hjorth_complexity"] = float(np.mean(comp_vals))
    row["eeg_permutation_entropy"] = float(np.mean(pe_vals))
    rows.append(row)

features_df = pd.DataFrame(rows)
t_center    = (w_start + w_end) / 2.0

print(f"Windows   : {len(features_df)}")
print(f"Features  : {list(features_df.columns)}")
features_df.describe().round(4)

## 4 · Build `coords_9d`

The 9D subcoordinate contract (v2.0) from `config_ingest_common_eeg.yaml → mnps_9d.versions.2.0`.

Each raw subcoordinate is normalised with `robust_z → clip(±6)`.
Band power features additionally go through `log10` first (as per `feature_standardization`).

In [ ]:
# ── Normalisation helpers ─────────────────────────────────────────────────────
ROBUST_SIGMA = 1.4826  # MAD → σ under normality
CLIP = 6.0

def _robust_z(v):
    """Robust-z normalise a 1-D array; return clipped result."""
    finite = v[np.isfinite(v)]
    if finite.size < 3:
        return np.zeros_like(v)
    med = np.median(finite)
    mad = np.median(np.abs(finite - med)) * ROBUST_SIGMA
    mad = mad if mad > 1e-9 else 1e-9
    return np.clip((v - med) / mad, -CLIP, CLIP)

def _log10_robust_z(v):
    """log10 → robust-z → clip for band-power columns."""
    safe = np.where(v > 0, v, 1e-20)
    return _robust_z(np.log10(safe))

# Pre-normalise feature columns used in 9D construction
LOG10_COLS = ["eeg_delta", "eeg_theta", "eeg_alpha", "eeg_beta", "eeg_gamma"]
RZ_COLS    = ["eeg_beta_alpha", "eeg_alpha_theta", "eeg_permutation_entropy",
              "eeg_hjorth_mobility", "eeg_hjorth_complexity"]

feat_norm = features_df.copy()
for col in LOG10_COLS:
    feat_norm[col] = _log10_robust_z(features_df[col].values)
for col in RZ_COLS:
    feat_norm[col] = _robust_z(features_df[col].values)

# ── 9D subcoordinate definitions (v2.0) ──────────────────────────────────────
NAMES_9D = ["m_a", "m_e", "m_o", "d_n", "d_l", "d_s", "e_e", "e_s", "e_m"]

# Weights from config mnps_9d.versions.2.0.subcoords
# e_m falls back to eeg_gamma (eeg_highfreq_power_30_45) per article fallback chain
V2_SUBCOORDS = {
    "m_a": {"eeg_delta": -0.5, "eeg_theta": -0.5},
    "m_e": {"eeg_alpha": -1.0},
    "m_o": {"eeg_beta_alpha": 1.0},
    "d_n": {"eeg_gamma": 1.0},
    "d_l": {"eeg_hjorth_mobility": 1.0},
    "d_s": {"eeg_alpha_theta": 1.0},
    "e_e": {"eeg_permutation_entropy": 1.0},
    "e_s": {"eeg_hjorth_complexity": 1.0},
    "e_m": {"eeg_gamma": 1.0},    # fallback: broadband γ as embodied proxy
}

coords_9d = np.zeros((N_WIN, 9), dtype=np.float32)
for i, sc_name in enumerate(NAMES_9D):
    val   = np.zeros(N_WIN, dtype=np.float64)
    w_abs = 0.0
    for feat, w in V2_SUBCOORDS[sc_name].items():
        if feat in feat_norm.columns:
            val   += feat_norm[feat].values * w
            w_abs += abs(w)
    coords_9d[:, i] = (val / w_abs if w_abs > 0 else val).astype(np.float32)

# Apply a final robust-z pass per subcoordinate (standard pipeline behaviour)
for i in range(9):
    coords_9d[:, i] = _robust_z(coords_9d[:, i].astype(np.float64)).astype(np.float32)

print(f"coords_9d shape  : {coords_9d.shape}")
print(f"Subcoord names   : {NAMES_9D}")
for i, name in enumerate(NAMES_9D):
    col = coords_9d[:, i]
    print(f"  {name:5s}  mean={col.mean():+.3f}  std={col.std():.3f}  finite={np.isfinite(col).all()}")

## 5 · Project `coords_9d` → `mnps_3d`

Fixed weighted projection `P` from `mnps_projection.v1_mapping`.  
Columns are **L2-normalised** at runtime before application (exact pipeline behaviour).

In [ ]:
# ── V1 mapping: 9D → 3D ──────────────────────────────────────────────────────
# From mnps_projection.v1_mapping in config_ingest_common_eeg.yaml
V1_MAPPING = {
    "m": {"m_a": 0.62, "m_e": 0.55, "m_o": 0.45},
    "d": {"d_n": 0.50, "d_l": 0.82, "d_s": 0.28},
    "e": {"e_e": 0.85, "e_s": 0.62, "e_m": 0.03},
}

x = np.zeros((N_WIN, 3), dtype=np.float32)
for axis_i, axis in enumerate(["m", "d", "e"]):
    # Build weight vector aligned to NAMES_9D ordering
    w_raw = np.array([V1_MAPPING[axis].get(n, 0.0) for n in NAMES_9D], dtype=np.float64)
    # L2-normalise (exact runtime behaviour)
    norm  = np.linalg.norm(w_raw)
    w_l2  = w_raw / (norm if norm > 1e-12 else 1.0)
    x[:, axis_i] = (coords_9d @ w_l2).astype(np.float32)

print(f"mnps_3d shape : {x.shape}")
print(f"  m  range  [{x[:,0].min():+.3f}, {x[:,0].max():+.3f}]")
print(f"  d  range  [{x[:,1].min():+.3f}, {x[:,1].max():+.3f}]")
print(f"  e  range  [{x[:,2].min():+.3f}, {x[:,2].max():+.3f}]")
print(f"All finite   : {np.isfinite(x).all()}")

## 6 · Derivatives, kNN, and Jacobians

Parameters from `mnps:` block in `config_ingest_common_eeg.yaml`:
- Derivative: Savitzky-Golay, `window=7`, `polyorder=3`
- kNN: `k=20`, `euclidean`, `whiten=True`
- Jacobian: `super_window=3`, `ridge_alpha=1.0`

In [ ]:
x_dot  = estimate_derivatives(x, dt=STEP_SEC, method="sav_gol", window=7, polyorder=3)
nn_idx = build_knn_indices(x, k=20, metric="euclidean", whiten=True)
jac    = estimate_local_jacobians(x, x_dot, nn_idx, super_window=3, ridge_alpha=1.0)
J_hat  = jac.j_hat

meta = compute_meta_indices(J_hat)
print(f"x_dot shape  : {x_dot.shape}")
print(f"J_hat shape  : {J_hat.shape}")
print(f"mean tr(J)   : {meta['mean_trace']:.4f}")
print(f"mean rot(J)  : {meta['mean_rotation_fro']:.4f}")

## 7 · Assemble `MNPSPayload`

The canonical container for all pipeline outputs.  
In a real run this is passed to the HDF5 writer; here we inspect it directly.

In [ ]:
REGIME_CODE = {"rest": 0, "drowsy": 1, "focused": 2, "alert": 3, "transition": 4}
stage = np.array([REGIME_CODE[r] for r in regime_per_win], dtype=np.int8)

payload = MNPSPayload(
    time         = t_center.astype(np.float64),
    x            = x,
    x_dot        = x_dot,
    stage        = stage,
    window_start = w_start.astype(np.float32),
    window_end   = w_end.astype(np.float32),
    coords_9d    = coords_9d,
    coords_9d_names = NAMES_9D,
    jacobian        = J_hat,
    jacobian_dot    = jac.j_dot,
    jacobian_centers= jac.centers,
    nn_indices   = nn_idx,
    attrs={"window_sec": WIN_SEC, "step_sec": STEP_SEC, "n_channels": N_CH,
           "dataset": "synthetic_quickstart", "fs": FS,
           "mnps_9d_version": "2.0", "mnps_3d_mode": "from_v2"},
)

print("MNPSPayload ready")
print(f"  time        : {payload.time.shape}")
print(f"  x (mnps_3d) : {payload.x.shape}")
print(f"  coords_9d   : {payload.coords_9d.shape}")
print(f"  jacobian    : {payload.jacobian.shape}")
print(f"  stages      : {list(REGIME_CODE.keys())} → codes {np.unique(stage)}")

## 8 · 3D MNPS trajectory plot

Coloured by brain-state regime. Hover over points to see timestamps and coordinates.

In [ ]:
PALETTE = {
    "rest":       "#4C9BE8",
    "drowsy":     "#A78BFA",
    "focused":    "#34D399",
    "alert":      "#F87171",
    "transition": "#FBBF24",
}

m_vals = payload.x[:, 0]
d_vals = payload.x[:, 1]
e_vals = payload.x[:, 2]
labels = np.array(regime_per_win)

if _PLOTLY:
    fig = go.Figure()
    fig.add_trace(go.Scatter3d(
        x=m_vals, y=d_vals, z=e_vals,
        mode="lines", line=dict(color="#e5e7eb", width=1.5),
        showlegend=False, name="trajectory",
    ))
    for regime, color in PALETTE.items():
        mask = labels == regime
        if not mask.any(): continue
        hover = [f"t={t:.1f}s | {regime}" for t in payload.time[mask]]
        fig.add_trace(go.Scatter3d(
            x=m_vals[mask], y=d_vals[mask], z=e_vals[mask],
            mode="markers",
            marker=dict(size=5, color=color, opacity=0.85,
                        line=dict(width=0.5, color="white")),
            name=regime, text=hover,
            hovertemplate="%{text}<br>m=%{x:.3f}  d=%{y:.3f}  e=%{z:.3f}<extra></extra>",
        ))
    for idx, lbl, sym in [(0, "start", "diamond"), (-1, "end", "square")]:
        fig.add_trace(go.Scatter3d(
            x=[m_vals[idx]], y=[d_vals[idx]], z=[e_vals[idx]],
            mode="markers+text",
            marker=dict(size=10, symbol=sym, color="black"),
            text=[lbl], textposition="top center", showlegend=False,
        ))
    fig.update_layout(
        title=dict(text="MNPS 3D trajectory  (coords_9d v2.0 → v1_mapping)",
                   x=0.5, xanchor="center"),
        scene=dict(
            xaxis_title="m  (Metastability)",
            yaxis_title="d  (Dynamics)",
            zaxis_title="e  (Entropy/Energy)",
            camera=dict(eye=dict(x=1.5, y=1.5, z=0.8)),
        ),
        legend=dict(title="Brain state", itemsizing="constant"),
        width=900, height=650, margin=dict(l=0, r=0, t=60, b=0),
    )
    fig.show()
else:
    fig = plt.figure(figsize=(10, 7))
    ax  = fig.add_subplot(111, projection="3d")
    ax.plot(m_vals, d_vals, e_vals, color="#e5e7eb", lw=0.8, zorder=1)
    for regime, color in PALETTE.items():
        mask = labels == regime
        if not mask.any(): continue
        ax.scatter(m_vals[mask], d_vals[mask], e_vals[mask],
                   c=color, s=30, label=regime, alpha=0.85, zorder=2)
    ax.set_xlabel("m  (Metastability)")
    ax.set_ylabel("d  (Dynamics)")
    ax.set_zlabel("e  (Entropy/Energy)")
    ax.set_title("MNPS 3D trajectory  (coords_9d v2.0 → v1_mapping)")
    ax.legend(title="Brain state", fontsize=8)
    plt.tight_layout()
    plt.savefig("mnps_quickstart_3d.png", dpi=150)
    plt.show()

## 9 · coords_9d heatmap — within-regime subcoordinate profiles

This shows **why the 9D chart matters**:  
regimes that look similar in 3D can have very different subcoordinate profiles.

In [ ]:
regime_means = {}
for regime in PALETTE:
    mask = labels == regime
    if mask.any():
        regime_means[regime] = coords_9d[mask].mean(axis=0)

mat = np.stack([regime_means[r] for r in PALETTE if r in regime_means])
row_labels = [r for r in PALETTE if r in regime_means]

if _PLOTLY:
    fig3 = go.Figure(go.Heatmap(
        z=mat, x=NAMES_9D, y=row_labels,
        colorscale="RdBu", zmid=0,
        colorbar=dict(title="robust-z"),
        hovertemplate="regime=%{y}<br>subcoord=%{x}<br>mean=%{z:.3f}<extra></extra>",
    ))
    fig3.update_layout(
        title="Mean coords_9d per regime  (contract v2.0)",
        xaxis_title="9D subcoordinate",
        yaxis_title="Brain state",
        width=750, height=320, margin=dict(l=80, r=20, t=50, b=60),
    )
    fig3.show()
else:
    fig3, ax3 = plt.subplots(figsize=(10, 3))
    im = ax3.imshow(mat, cmap="RdBu", aspect="auto",
                    vmin=-np.abs(mat).max(), vmax=np.abs(mat).max())
    ax3.set_xticks(range(9)); ax3.set_xticklabels(NAMES_9D)
    ax3.set_yticks(range(len(row_labels))); ax3.set_yticklabels(row_labels)
    plt.colorbar(im, ax=ax3, label="robust-z")
    ax3.set_title("Mean coords_9d per regime  (contract v2.0)")
    plt.tight_layout()
    plt.show()

## 10 · Jacobian trace over time

In [ ]:
j_trace   = np.trace(J_hat, axis1=1, axis2=2)
j_times   = payload.time[jac.centers]
j_regimes = labels[jac.centers]

if _PLOTLY:
    fig4 = go.Figure()
    for regime, color in PALETTE.items():
        mask = j_regimes == regime
        if not mask.any(): continue
        fig4.add_trace(go.Scatter(
            x=j_times[mask], y=j_trace[mask],
            mode="markers+lines",
            marker=dict(size=6, color=color), line=dict(color=color, width=1),
            name=regime,
        ))
    fig4.add_hline(y=0, line_dash="dash", line_color="gray", opacity=0.5)
    fig4.update_layout(
        title="Jacobian trace  (expansion > 0, contraction < 0)",
        xaxis_title="Time (s)", yaxis_title="tr(J)",
        width=900, height=340, margin=dict(l=0, r=0, t=50, b=0),
    )
    fig4.show()
else:
    fig4, ax4 = plt.subplots(figsize=(11, 3))
    for regime, color in PALETTE.items():
        mask = j_regimes == regime
        if not mask.any(): continue
        ax4.scatter(j_times[mask], j_trace[mask], c=color, s=20, label=regime, zorder=3)
    ax4.axhline(0, ls="--", color="gray", alpha=0.5)
    ax4.set_xlabel("Time (s)"); ax4.set_ylabel("tr(J)")
    ax4.set_title("Jacobian trace")
    ax4.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 11 · Per-regime summary

In [ ]:
summary_rows = []
for regime in PALETTE:
    mw = labels == regime
    mj = j_regimes == regime
    if not mw.any(): continue
    summary_rows.append({
        "regime":     regime,
        "n_windows":  int(mw.sum()),
        "m_mean":     round(float(m_vals[mw].mean()), 3),
        "d_mean":     round(float(d_vals[mw].mean()), 3),
        "e_mean":     round(float(e_vals[mw].mean()), 3),
        "tr_J_mean":  round(float(j_trace[mj].mean()), 4) if mj.any() else float("nan"),
    })

summary = pd.DataFrame(summary_rows).set_index("regime")
print(summary.to_string())

---

## What next?

| Step | How |
|------|-----|
| **Run on real EEG** | `python -m mndm.cli all --dataset ds004511 --config mndm/config/config_ingest_ds004511.yaml` |
| **Read HDF5 output** | `import h5py; f = h5py.File('summary.h5'); mnps = f['/mnps_3d'][:]; c9d = f['/coords_9d/values'][:]` |
| **Enable 9D Jacobians** | `mnps.jacobian.enabled: true` (already default) |
| **Embodied anchoring** | `anchor_state` / `anchor_quality` exports via MNDM 2.3 anchoring config |
| **Read the docs** | https://neuralmanifolddynamics.readthedocs.io |

---

*NeuralManifoldDynamics 2.3 — a versioned measurement contract for low-dimensional neural-manifold trajectories.*